In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date, lower, trim

In [4]:
spark = SparkSession.builder \
    .appName("VenchiDataLoading") \
    .getOrCreate()

## Read Data 2

In [5]:
# Define Paths
accounts_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\accounts.parquet'
interactions_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\interactions.parquet'
products_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\products.txt'
sales_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\sales.parquet'

# 1. Accounts (Parquet preserves schema, but we ensure timestamp type)
print("Accounts")
accounts_df = spark.read.parquet(accounts_path) \
    .withColumn("timestamp", to_timestamp(col("timestamp")))
accounts_df.printSchema()

# 2. Interactions
print("Interactions")
interactions_df = spark.read.parquet(interactions_path) \
    .withColumn("date", to_date(col("date")))
interactions_df.printSchema()

# 3. Sales
print("Sales")
sales_df = spark.read.parquet(sales_path) \
    .withColumn("date", to_date(col("date")))
sales_df.printSchema()

# 4. Products (Reading Tab-Separated TXT)
print("Products")
products_df = spark.read.csv(products_path, sep='\t', header=True, inferSchema=True)
products_df.printSchema()

Accounts
root
 |-- account_id: string (nullable = true)
 |-- hct: long (nullable = true)
 |-- staff: long (nullable = true)
 |-- turnover_m_usd: long (nullable = true)
 |-- brand_loyalty: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)

Interactions
root
 |-- interaction_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- duration_mins: long (nullable = true)
 |-- response: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

Sales
root
 |-- sale_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

Products
root
 |-- product_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



## Task 1

### Data Cleaning

In [6]:
print(accounts_df.count())
accounts_clean = accounts_df.filter(col("account_id").isNotNull())
print(accounts_clean.count())

12205
12205


In [7]:
accounts_clean = accounts_df.filter(col("account_id").isNotNull())

interactions_clean = interactions_df.filter(col("account_id").isNotNull()) \
    .withColumn("topic_clean", lower(trim(col("topic")))) \
    .withColumn("channel_clean", lower(trim(col("channel")))) \
    .withColumn("response_clean", lower(trim(col("response"))))

sales_clean = sales_df.filter(col("account_id").isNotNull())
products_clean = products_df.filter(col("product_id").isNotNull())

### KPI 1: Total Revenue Generated

In [8]:
from pyspark.sql import functions as F

In [13]:
products_df.show(5)

+--------------------+-----------+-----+------------+--------------------+
|          product_id|   maturity|price|    category|           patent_id|
+--------------------+-----------+-----+------------+--------------------+
|8a558cf9e24a83edd...|established|  812|        feed|47385e093281de466...|
|cf682b73be1afad47...|        new| 3654|       seeds|d139f78e2d2d7544a...|
|f26b0d8f252a76f2f...|        new| 4901|       seeds|a33adb5c233e93ae9...|
|0d98dad3689d87a13...|established| 5347|anti-biotics|96b2fb148ff71d24c...|
|4210ca147391331cc...|established|  716|anti-biotics|0cfd140809104bb01...|
+--------------------+-----------+-----+------------+--------------------+
only showing top 5 rows


In [16]:
sales_with_price = sales_clean.join(products_clean, on="product_id", how="left")
sales_with_price.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



In [11]:
# Aggregate revenue per customer
kpi_revenue = sales_with_price.groupBy("account_id", "product_id").agg(
    F.sum("price").alias("total_revenue")
)

In [12]:
kpi_revenue.show(10)

+--------------------+--------------------+-------------+
|          account_id|          product_id|total_revenue|
+--------------------+--------------------+-------------+
|b054d4869d9e11e91...|a1f83487893c2d96c...|         2529|
|93886fd4d75acd112...|af4cd8bcdc61421aa...|         4157|
|d6c89f029ba55b9d6...|b88653f7ff85b3d35...|         1542|
|9115794a5216927f0...|f3bff95e49ab72d76...|         7409|
|a91d016f0ffb42084...|c4bffcfdc62b40169...|         6479|
|b86f757259a84fecc...|ce4c9cc1ce93dc31b...|         1972|
|dab526b2af5d7c589...|01dc0afbf4732dbbd...|         7628|
|c996cc0bb0f1a1c85...|e744f0bc384a80ee1...|          457|
|7ce7bb825f1f0db54...|af4cd8bcdc61421aa...|         4157|
|b47b0470ed37839d8...|34319830ed90520be...|         3602|
+--------------------+--------------------+-------------+
only showing top 10 rows


In [17]:
kpi_revenue_category = sales_with_price.groupBy("account_id", "category").agg(
    F.sum("price").alias("total_revenue")
)

In [18]:
kpi_revenue_category.show(10)

+--------------------+------------+-------------+
|          account_id|    category|total_revenue|
+--------------------+------------+-------------+
|4824b7c04d01b9722...|       seeds|        19541|
|c06fe30e1d32007e5...|        feed|        28668|
|3dbbbe2c20f31be46...|  fertiliser|        18833|
|fcc8bf70d6034d1ce...|        feed|        24497|
|0f5133ff921f4bffd...|       seeds|        25601|
|eaf42508d48b40ce6...|anti-biotics|        27523|
|f9722bf5f08ff4d22...|  fertiliser|        38683|
|d39a02df05ca16d18...|        feed|        19909|
|d3a00a97d4b55c60a...|anti-biotics|        33678|
|a2dd04fcb5cd88578...|        feed|        24713|
+--------------------+------------+-------------+
only showing top 10 rows


### KPI 2: Number of sales interactions in the last 6 months

In [ ]:
max_date_row = interactions_clean.select(F.max("date").alias("latest_date")).collect()[0]
latest_date = max_date_row["latest_date"]
print(f"Latest interaction date: {latest_date}")

Latest interaction date: 2021-02-05


In [ ]:
sales_interactions_6m = interactions_clean.filter(
    (F.col("date") >= F.date_sub(F.lit(latest_date), 180))
)
sales_interactions_6m.show(10)

+--------------------+------------+-------------+--------+-------------------+----------+--------------------+--------------------+-------------------+-------------+--------------+
|      interaction_id|     channel|duration_mins|response|              topic|      date|          account_id|          product_id|        topic_clean|channel_clean|response_clean|
+--------------------+------------+-------------+--------+-------------------+----------+--------------------+--------------------+-------------------+-------------+--------------+
|9e8a83833a5a5dafb...|       email|            6|positive|against competition|2020-10-17|544feaa1526ef94f1...|2a38ce238f4b3a52b...|against competition|        email|      positive|
|17f65a62b55523036...|face to face|           82|   mixed|  available finance|2021-01-24|f63ebf8ff4a382c2e...|0eed3f6963570b036...|  available finance| face to face|         mixed|
|909f3a6eca8fbf6b7...|face to face|           55|positive|               cost|2020-09-22|149a85

In [23]:
# Aggregate count per customer
kpi_recent_sales_interactions = sales_interactions_6m.groupBy("account_id").agg(
    F.count("interaction_id").alias("sales_interactions_last_6m")
)

In [24]:
kpi_recent_sales_interactions.show(10)

+--------------------+--------------------------+
|          account_id|sales_interactions_last_6m|
+--------------------+--------------------------+
|ae9e95dc1fb78abec...|                         1|
|2b399e65042a6c1b5...|                         1|
|fb8a2c3a0a5044235...|                         1|
|4029ac1cf58212d04...|                         1|
|3cb614754aec72774...|                         1|
|2399c7d6fd5d8a213...|                         1|
|30c501d1b5b4b8fce...|                         1|
|309dae2ef6410ce4a...|                         1|
|7ae20648c2c5e521c...|                         1|
|fdaf5d1016c282454...|                         1|
+--------------------+--------------------------+
only showing top 10 rows
